In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. ROBUST PATH SETUP ---
# This list covers all likely folder structures we've discussed
possible_paths = [
    '../yfinance_data/', 
    './yfinance_data/', 
    '../data/yfinance_data/',
    r'C:\Users\HP\Desktop\10acadamey\news-sentiment-analysis\yfinance_data'
]

stock_data_path = None
for p in possible_paths:
    if os.path.exists(p):
        stock_data_path = p
        print(f"✅ Found data directory at: {p}")
        break

if not stock_data_path:
    print("❌ ERROR: Could not find the 'yfinance_data' folder.")
    print("Please ensure the folder is in your project directory or update the path manually.")

# --- 2. DATA LOADING & PREPARATION FUNCTION ---
def load_and_clean_stock_data(directory):
    all_stocks = {}
    
    # Loop through every CSV in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            # Extract ticker name (e.g., AAPL from AAPL_stock_data.csv)
            ticker = filename.split('_')[0].upper()
            file_path = os.path.join(directory, filename)
            
            # Load DataFrame
            df = pd.read_csv(file_path)
            
            # A. Ensure Columns are correctly typed
            # Convert Date to datetime and set as index
            if 'Date' in df.columns:
                df['Date'] = pd.to_datetime(df['Date'], utc=True)
                df.set_index('Date', inplace=True)
            
            # B. Standardize numeric columns (Open, High, Low, Close, Volume)
            target_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            for col in target_cols:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
            
            # C. Handle Missing Values
            # Financial data uses interpolation/filling to maintain time-series continuity
            if df.isnull().values.any():
                df = df.sort_index() # Ensure chronological order before filling
                df = df.interpolate(method='time').ffill().bfill()
            
            all_stocks[ticker] = df
            print(f"📊 Processed {ticker}: {len(df)} trading days.")
            
    return all_stocks

# --- 3. EXECUTION ---
if stock_data_path:
    stocks_dict = load_and_clean_stock_data(stock_data_path)
    
    # Quick Check on the first available stock
    if stocks_dict:
        sample_ticker = list(stocks_dict.keys())[0]
        print(f"\n--- 🧪 DATA QUALITY CHECK: {sample_ticker} ---")
        print(stocks_dict[sample_ticker].info())
        print("\nHead of Data:")
        print(stocks_dict[sample_ticker].head())
    else:
        print("⚠ The directory was found but no .csv files were detected inside.")

ModuleNotFoundError: No module named 'seaborn'

In [3]:
import talib

def apply_technical_indicators(stocks_dict):
    for ticker, df in stocks_dict.items():
        # Ensure data is sorted by date for rolling calculations
        df = df.sort_index()

        # 1. Simple Moving Averages (20-day and 50-day)
        df['SMA_20'] = talib.SMA(df['Close'], timeperiod=20)
        df['SMA_50'] = talib.SMA(df['Close'], timeperiod=50)

        # 2. RSI (Relative Strength Index)
        df['RSI'] = talib.RSI(df['Close'], timeperiod=14)

        # 3. MACD
        df['MACD'], df['MACD_signal'], df['MACD_hist'] = talib.MACD(
            df['Close'], fastperiod=12, slowperiod=26, signalperiod=9
        )
        
        # Save changes back to the dictionary
        stocks_dict[ticker] = df
        print(f"✅ Calculated indicators for {ticker}")

# Run the calculation
apply_technical_indicators(stocks_dict)

# Quick check: View the tail of a stock to see the new columns
sample_ticker = list(stocks_dict.keys())[0]
print(f"\n--- New columns in {sample_ticker} ---")
print(stocks_dict[sample_ticker][['Close', 'SMA_20', 'RSI', 'MACD']].tail())

ModuleNotFoundError: No module named 'talib'

In [6]:
def plot_stock_trends(ticker):
    df = stocks_dict[ticker].tail(200) # Plot last 200 days for clarity
    
    plt.figure(figsize=(14, 7))
    plt.plot(df.index, df['Close'], label='Close Price', alpha=0.5)
    plt.plot(df.index, df['SMA_20'], label='SMA 20', color='orange')
    plt.plot(df.index, df['SMA_50'], label='SMA 50', color='green')
    
    plt.title(f'{ticker} Stock Price and Moving Averages')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.show()

# Visualize one stock (e.g., TSLA)
plot_stock_trends('TSLA')

NameError: name 'stocks_dict' is not defined